# Notebook 03: GRACE/GRACE-FO + GLDAS Groundwater Storage Anomaly
**Pipeline Stage 5 — Groundwater Storage Estimation**

Estimates monthly groundwater storage anomalies (GWSA) for coastal Ghana:

> **GWSA = TWSA − SMSA − CWSA**

- **TWSA** = Total water storage anomaly (JPL mascon RL06.3, cm → mm EWH)
- **SMSA** = Soil moisture storage anomaly (GLDAS NOAH 0.25° v2.1, kg m⁻²)
- **CWSA** = Canopy water storage anomaly (GLDAS, kg m⁻²)

Snow/ice excluded (tropical Ghana).

> ⚠️ GRACE spatial resolution ≈ 300–400 km. All analysis is regional/basin-scale.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore', category=FutureWarning)

import os
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from pathlib import Path

from src.config import paths, rp
from src.utils import setup_logging, ensure_dir
from src.groundwater import (
    run_mann_kendall_trend, compute_sens_slope, classify_groundwater_drought,
)

log = setup_logging()
plt.rcParams.update({'figure.dpi': 150, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})
print('Imports OK')

In [ ]:
# ── Paths and constants ─────────────────────────────────────────────────────
GRACE_FILE = rp('data/raw/grace/GRCTellus.JPL.200204_202604.GLO.RL06.3M.MSCNv04CRI.nc')
GLDAS_DIR  = rp('data/raw/gldas')
OUT_TS     = rp('data/interim/time_series')
OUT_GW     = rp('data/processed/groundwater_anomalies')
FIG_DIR    = rp('outputs/figures')
for d in [OUT_TS, OUT_GW, FIG_DIR]:
    ensure_dir(d)

GHANA_LAT  = slice(4.0, 11.5)
GHANA_LON  = slice(-3.5, 1.5)
BASELINE   = slice('2004-01-01', '2009-12-31')
SM_VARS    = ['SoilMoi0_10cm_inst', 'SoilMoi10_40cm_inst',
              'SoilMoi40_100cm_inst', 'SoilMoi100_200cm_inst']
CANOPY_VAR = 'CanopInt_inst'

DROUGHT_PALETTE = {
    'Wet': '#4393c3', 'Mildly wet': '#92c5de', 'Near normal': '#f7f7f7',
    'Mild drought': '#fddbc7', 'Moderate drought': '#f4a582',
    'Severe drought': '#d6604d', 'Extreme drought': '#b2182b',
}
print('Config ready. GLDAS files:', len(list(GLDAS_DIR.glob('*.nc4'))))

In [ ]:
# ── 1. Load GRACE TWSA ──────────────────────────────────────────────────────
# GRACE uses 0-360° lon — convert to -180/180 before selecting Ghana
ds_raw = xr.open_dataset(GRACE_FILE)
ds = ds_raw.assign_coords(lon=((ds_raw.lon + 180) % 360 - 180)).sortby('lon')
twsa_grid = (
    ds['lwe_thickness'].sel(lat=GHANA_LAT, lon=GHANA_LON)
    * ds['scale_factor'].sel(lat=GHANA_LAT, lon=GHANA_LON)
    * ds['land_mask'].sel(lat=GHANA_LAT, lon=GHANA_LON)
) * 10.0   # cm → mm EWH

twsa_raw_series = twsa_grid.mean(dim=['lat', 'lon'], skipna=True).to_series()
twsa_raw_series.index = twsa_raw_series.index.to_period('M').to_timestamp('M', how='start')
twsa_raw_series = twsa_raw_series[~twsa_raw_series.index.duplicated(keep='first')]
twsa_raw_series.name = 'twsa_mm'
ds.close()

print(f'GRACE TWSA: {len(twsa_raw_series)} months  '
      f'({twsa_raw_series.index[0].strftime("%Y-%m")} → '
      f'{twsa_raw_series.index[-1].strftime("%Y-%m")})')
print(f'  NaN months (mission gap + missing): {twsa_raw_series.isna().sum()}')
twsa_raw_series.iloc[:5]

In [ ]:
# ── 2. Load GLDAS soil moisture and canopy water ────────────────────────────
# Materialise GLDAS arrays into memory (.compute()) so they can be reused
# in both the spatial-mean time series AND the gridded product (cell 10)
# without reloading 264 files a second time.
print('Opening and computing GLDAS arrays into memory ...')
gldas_files = sorted(GLDAS_DIR.glob('*.nc4'))
ds_g = xr.open_mfdataset(
    [str(f) for f in gldas_files],
    combine='by_coords',
    parallel=False,
)

sm_total = sum(ds_g[v].sel(lat=GHANA_LAT, lon=GHANA_LON) for v in SM_VARS)
canopy   = ds_g[CANOPY_VAR].sel(lat=GHANA_LAT, lon=GHANA_LON)

# Materialise — after this, ds_g can be closed
sm_g_da  = sm_total.compute()   # shape: (time, lat, lon), kept in memory
can_g_da = canopy.compute()
ds_g.close()

def to_monthly_series(da, name):
    s = da.mean(dim=['lat', 'lon'], skipna=True).to_series()
    s.index = s.index.to_period('M').to_timestamp('M', how='start')
    s = s[~s.index.duplicated(keep='first')]
    s.name = name
    return s

sm_series  = to_monthly_series(sm_g_da, 'sm_mm')
can_series = to_monthly_series(can_g_da, 'canopy_mm')

print(f'GLDAS: {len(sm_series)} months  '
      f'({sm_series.index[0].strftime("%Y-%m")} → '
      f'{sm_series.index[-1].strftime("%Y-%m")})')
print(f'  SM mean: {sm_series.mean():.1f} mm EWH')
print(f'  Canopy mean: {can_series.mean():.4f} mm EWH')

In [ ]:
# ── 3. Compute storage anomalies (2004–2009 baseline) ───────────────────────
def monthly_anomaly(series: pd.Series, baseline: slice) -> pd.Series:
    """Remove monthly climatological mean computed over baseline period."""
    base = series.loc[baseline]
    clim = base.groupby(base.index.month).mean()
    anom = series - series.index.to_series().apply(lambda d: clim.get(d.month, np.nan))
    return anom

twsa_anom = monthly_anomaly(twsa_raw_series, BASELINE)
sm_anom   = monthly_anomaly(sm_series,       BASELINE)
can_anom  = monthly_anomaly(can_series,      BASELINE)

twsa_anom.name = 'twsa_anom_mm'
sm_anom.name   = 'smsa_mm'
can_anom.name  = 'cwsa_mm'

# ── 4. GWSA = TWSA - SMSA - CWSA ────────────────────────────────────────────
df = pd.concat([twsa_anom, sm_anom, can_anom], axis=1, sort=True)
df['gwsa_mm'] = df['twsa_anom_mm'] - df['smsa_mm'] - df['cwsa_mm']
df = df.dropna(subset=['twsa_anom_mm'])

gwsa_mu, gwsa_std = df['gwsa_mm'].mean(), df['gwsa_mm'].std()
df['sgwa'] = (df['gwsa_mm'] - gwsa_mu) / gwsa_std

print('Water balance components (mm EWH):')
print(df[['twsa_anom_mm', 'smsa_mm', 'cwsa_mm', 'gwsa_mm', 'sgwa']].describe().round(2))

In [ ]:
# ── 5. Drought classification ───────────────────────────────────────────────
def classify_drought(sgwa: float) -> str:
    if   sgwa >=  1.0: return 'Wet'
    elif sgwa >=  0.5: return 'Mildly wet'
    elif sgwa >= -0.5: return 'Near normal'
    elif sgwa >= -1.0: return 'Mild drought'
    elif sgwa >= -1.5: return 'Moderate drought'
    elif sgwa >= -2.0: return 'Severe drought'
    else:              return 'Extreme drought'

df['drought_class'] = df['sgwa'].apply(classify_drought)
print('Drought class distribution:')
print(df['drought_class'].value_counts().sort_values(ascending=False).to_string())

In [ ]:
# ── 6. Save interim CSVs ────────────────────────────────────────────────────
twsa_raw_series.to_csv(OUT_TS / 'grace_twsa_monthly.csv', header=True)
sm_series.to_csv(OUT_TS       / 'gldas_soil_moisture_monthly.csv', header=True)
can_series.to_csv(OUT_TS      / 'gldas_canopy_water_monthly.csv', header=True)
df.to_csv(OUT_TS              / 'gwsa_monthly.csv')
print('CSVs saved to', OUT_TS)

In [ ]:
# ── 7. GRACE TWSA time series figure ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))

colors = np.where(twsa_anom >= 0, '#4393c3', '#d6604d')
ax.bar(twsa_anom.index, twsa_anom.values, color=colors, alpha=0.75, width=25)
ax.axhline(0, color='#333333', lw=0.8, ls='--')
ax.axvspan(pd.Timestamp('2017-07'), pd.Timestamp('2018-06'),
           color='#bdbdbd', alpha=0.45, label='GRACE mission gap')
roll = twsa_anom.rolling(12, center=True, min_periods=6).mean()
ax.plot(roll.index, roll.values, color='#252525', lw=2.0, label='12-month rolling mean')

ax.set_title('GRACE/GRACE-FO Total Water Storage Anomaly — Coastal Ghana\n'
             '(JPL mascon RL06.3, CRI-filtered, 2004–2009 baseline)', fontsize=11, pad=10)
ax.set_ylabel('TWSA (mm EWH)', fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=9)
ax.set_xlim(twsa_anom.dropna().index[0], twsa_anom.dropna().index[-1])
fig.text(0.5, -0.03,
    'JPL GRACE/GRACE-FO Mascon RL06.3Mv04 | Scale factor + land mask applied | '
    'Spatial mean over coastal Ghana 50 km buffer',
    ha='center', fontsize=7, color='#636363')
plt.tight_layout()
plt.savefig(FIG_DIR / 'grace_twsa_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8. GWSA + drought classification figure ─────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True,
                                gridspec_kw={'height_ratios': [2.5, 1]})

col_gwsa = np.where(df['gwsa_mm'] >= 0, '#4393c3', '#d6604d')
ax1.bar(df.index, df['gwsa_mm'], color=col_gwsa, alpha=0.75, width=25)
ax1.axhline(0, color='#333333', lw=0.8, ls='--')
ax1.axvspan(pd.Timestamp('2017-07'), pd.Timestamp('2018-06'),
            color='#bdbdbd', alpha=0.4, label='GRACE mission gap')
roll_gw = df['gwsa_mm'].rolling(12, center=True, min_periods=6).mean()
ax1.plot(roll_gw.index, roll_gw.values, color='#252525', lw=2.0, label='12-month rolling mean')
ax1.set_ylabel('GWSA (mm EWH)', fontsize=10)
ax1.set_title('Groundwater Storage Anomaly — Coastal Ghana\n'
              '(GWSA = TWSA − SMSA − CWSA | 2004–2009 baseline)', fontsize=11, pad=10)
ax1.legend(fontsize=9)

for dt, row in df.iterrows():
    ax2.bar(dt, row['sgwa'],
            color=DROUGHT_PALETTE.get(row['drought_class'], 'grey'),
            alpha=0.85, width=25)
ax2.axhline(0, color='#333333', lw=0.8, ls='--')
for t in [-0.5, -1.0, -1.5, -2.0]:
    ax2.axhline(t, color='#969696', lw=0.5, ls=':')
ax2.set_ylabel('Std. GWSA\n(SGWA)', fontsize=9)
ax2.legend(handles=[Patch(color=v, label=k) for k, v in DROUGHT_PALETTE.items()],
           ncol=4, fontsize=7.5, loc='lower left', framealpha=0.9)
ax2.xaxis.set_major_locator(mdates.YearLocator(2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.tick_params(axis='x', rotation=45)
ax2.set_xlim(df.index[0], df.index[-1])
fig.text(0.5, -0.01,
    'GRACE JPL RL06.3 + GLDAS NOAH v2.1 | Spatial mean, coastal Ghana 50 km buffer',
    ha='center', fontsize=7, color='#636363')
plt.tight_layout()
plt.savefig(FIG_DIR / 'gwsa_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── 9. Mann-Kendall trend test ───────────────────────────────────────────────
try:
    gwsa_clean = df['gwsa_mm'].dropna()
    mk  = run_mann_kendall_trend(gwsa_clean, method='original')
    sen = compute_sens_slope(gwsa_clean)
    direction = 'declining' if sen < 0 else 'increasing'
    sig = 'significant' if mk.p < 0.05 else 'not significant'
    print(f'Mann-Kendall trend test on GWSA ({len(gwsa_clean)} months):')
    print(f'  Trend    : {mk.trend}  (p = {mk.p:.4f}) — {sig} at α=0.05')
    print(f'  Tau      : {mk.Tau:.4f}')
    print(f"  Sen slope: {sen:.3f} mm/month  ({sen*12:.2f} mm/year)")
    print()
    print(f'  → Groundwater storage shows a {direction} trend of '
          f'{abs(sen*12):.2f} mm/year ({sig}).')
except Exception as e:
    print(f'pymannkendall not available: {e}')

In [ ]:
# ── 10. Save gridded GWSA NetCDF ─────────────────────────────────────────────
# sm_g_da and can_g_da were materialised in cell 2 — no GLDAS reload needed.

print('Building gridded GWSA product ...')

def normalise_time(da: xr.DataArray) -> xr.DataArray:
    times_ms = pd.DatetimeIndex(da.time.values).to_period('M').to_timestamp()
    _, keep = np.unique(times_ms, return_index=True)
    return da.isel(time=keep).assign_coords(time=('time', times_ms[keep]))

def xr_monthly_anomaly(da: xr.DataArray, baseline) -> xr.DataArray:
    clim = da.sel(time=baseline).groupby('time.month').mean('time')
    return da.groupby('time.month') - clim

# GRACE gridded anomaly — convert 0-360 lon to -180/180 first
ds_grace_raw = xr.open_dataset(GRACE_FILE)
ds_grace = ds_grace_raw.assign_coords(
    lon=((ds_grace_raw.lon + 180) % 360 - 180)
).sortby('lon')
twsa_g = (
    ds_grace['lwe_thickness'].sel(lat=GHANA_LAT, lon=GHANA_LON)
    * ds_grace['scale_factor'].sel(lat=GHANA_LAT, lon=GHANA_LON)
    * ds_grace['land_mask'].sel(lat=GHANA_LAT, lon=GHANA_LON)
) * 10.0
twsa_g = normalise_time(twsa_g)
twsa_anom_g = xr_monthly_anomaly(twsa_g, BASELINE)
ds_grace.close()

# GLDAS gridded anomaly — reuse already-computed arrays from cell 2
sm_g_norm  = normalise_time(sm_g_da)
can_g_norm = normalise_time(can_g_da)
sm_anom_g  = xr_monthly_anomaly(sm_g_norm,  BASELINE)
can_anom_g = xr_monthly_anomaly(can_g_norm, BASELINE)

# Regrid GLDAS 0.25° → GRACE 0.5°, then inner-join on time
sm_r  = sm_anom_g.interp(lat=twsa_anom_g.lat, lon=twsa_anom_g.lon, method='linear')
can_r = can_anom_g.interp(lat=twsa_anom_g.lat, lon=twsa_anom_g.lon, method='linear')
twsa_a, sm_a, can_a = xr.align(twsa_anom_g, sm_r, can_r, join='inner')
print(f'Common time steps: {len(twsa_a.time)}')

gwsa_g = twsa_a - sm_a - can_a
gwsa_g.name = 'gwsa'
gwsa_g.attrs = {
    'units': 'mm EWH',
    'long_name': 'Groundwater storage anomaly',
    'baseline': '2004-01 to 2009-12',
    'method': 'GWSA = TWSA - SMSA - CWSA',
    'grace_product': 'TELLUS_GRAC-GRFO_MASCON_CRI_GRID_RL06.3_V4',
    'gldas_product': 'GLDAS_NOAH025_M v2.1',
}
ds_out = gwsa_g.to_dataset()
ds_out.attrs.update({'Conventions': 'CF-1.8',
                     'title': 'Coastal Ghana Groundwater Storage Anomaly'})
ds_out.to_netcdf(OUT_GW / 'gwsa_monthly_grid.nc')
print('Saved:', OUT_GW / 'gwsa_monthly_grid.nc')

In [ ]:
# ── 11. Save regional CSV and print summary ──────────────────────────────────
df[['twsa_anom_mm','smsa_mm','cwsa_mm','gwsa_mm','sgwa','drought_class']].to_csv(
    OUT_GW / 'gwsa_by_region_monthly.csv')
print('Regional GWSA CSV saved:', OUT_GW / 'gwsa_by_region_monthly.csv')

print()
print('=' * 55)
print('Stage 5 complete — outputs:')
for p in [
    OUT_TS  / 'grace_twsa_monthly.csv',
    OUT_TS  / 'gldas_soil_moisture_monthly.csv',
    OUT_TS  / 'gldas_canopy_water_monthly.csv',
    OUT_TS  / 'gwsa_monthly.csv',
    OUT_GW  / 'gwsa_monthly_grid.nc',
    OUT_GW  / 'gwsa_by_region_monthly.csv',
    FIG_DIR / 'grace_twsa_timeseries.png',
    FIG_DIR / 'gwsa_timeseries.png',
]:
    tick = '✓' if Path(p).exists() else '✗ MISSING'
    print(f'  {tick}  {Path(p).name}')